# 02 — Quantum Boltzmann Machine Training

Train a QBM and classical baselines (RBM, autoencoder) on agent
trajectory data. Compare reconstruction quality and latent
representations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.quantum_boltzmann import QuantumBoltzmannMachine, QBMConfig
from src.classical_baselines import ClassicalRBM, Autoencoder

## 1. Load data

In [ ]:
data = np.load('../data/agent_trajectories.npz', allow_pickle=True)
trajectories = data['trajectories']
labels = data['labels']
label_names = list(data['label_names'])

# Flatten to (N*T, 7) for RBM training
N, T, D = trajectories.shape
flat_data = trajectories.reshape(-1, D)
print(f'Flat training data: {flat_data.shape}')

## 2. Train QBM

In [ ]:
cfg = QBMConfig(
    n_visible=D, n_hidden=8, gamma=0.5, beta=1.0,
    learning_rate=0.01, cd_steps=1, n_epochs=60, batch_size=64,
)
qbm = QuantumBoltzmannMachine(cfg)
qbm.fit(flat_data, verbose=True)

## 3. Train classical baselines

In [ ]:
rbm = ClassicalRBM(n_visible=D, n_hidden=8, n_epochs=60, batch_size=64)
rbm.fit(flat_data, verbose=True)

ae = Autoencoder(n_input=D, n_bottleneck=8, n_encoder=32, n_epochs=100, batch_size=64)
ae.fit(flat_data, verbose=True)

## 4. Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(qbm.loss_history, label='QBM')
ax.plot(rbm.loss_history, label='Classical RBM')
ax.plot(ae.loss_history, label='Autoencoder')
ax.set_xlabel('Epoch')
ax.set_ylabel('Reconstruction Loss')
ax.set_title('Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Latent space visualisation (PCA of hidden activations)

In [ ]:
from sklearn.decomposition import PCA

# Encode mean trajectory per sample
mean_trajs = trajectories.mean(axis=1)  # (N, 7)
latent_qbm = qbm.encode(mean_trajs)
latent_rbm = rbm.encode(mean_trajs)
latent_ae = ae.encode(mean_trajs)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, Z) in zip(axes, [('QBM', latent_qbm), ('RBM', latent_rbm), ('AE', latent_ae)]):
    pca = PCA(n_components=2).fit_transform(Z)
    for li, ln in enumerate(label_names):
        mask = labels == li
        ax.scatter(pca[mask, 0], pca[mask, 1], s=10, alpha=0.5, label=ln)
    ax.set_title(f'{name} latent PCA')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/latent_space_viz.png', dpi=150)
plt.show()